# NaukriAI — CV Parser Fine-Tuning (Google Colab)

**Runtime required:** GPU (T4 or better)  
In Colab: **Runtime → Change runtime type → T4 GPU**

This notebook fine-tunes Mistral 7B using QLoRA on our 101-example CV parsing dataset.  
Output: a LoRA adapter + GGUF model you can load into Ollama.

In [ ]:
# Check GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE — switch runtime!"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# Install dependencies (takes ~3 minutes)
!pip install -q unsloth trl peft bitsandbytes datasets transformers accelerate

In [ ]:
# Upload dataset
# Option A: Upload from your PC
from google.colab import files
print('Upload ai-engine/finetuning/dataset_examples.jsonl from your PC')
uploaded = files.upload()  # click the button, select dataset_examples.jsonl

# Option B: Paste dataset path if already uploaded to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = '/content/drive/MyDrive/naukri/dataset_examples.jsonl'

In [ ]:
import json
from datasets import Dataset

DATASET_PATH = 'dataset_examples.jsonl'  # file uploaded above
OUTPUT_DIR   = '/content/naukri-cv-parser-lora'
MODEL_NAME   = 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit'

# Load dataset
examples = []
with open(DATASET_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            ex = json.loads(line)
            instruction = ex['instruction']
            inp = ex.get('input', '')
            output = ex['output']
            if inp:
                text = f'[INST] {instruction}\n\n{inp} [/INST] {output}'
            else:
                text = f'[INST] {instruction} [/INST] {output}'
            examples.append({'text': text})

dataset = Dataset.from_list(examples)
print(f'Loaded {len(dataset)} training examples')
print('Sample:', dataset[0]['text'][:200])

In [ ]:
from unsloth import FastLanguageModel

print(f'Loading {MODEL_NAME}...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=4096,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        max_steps=200,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type='cosine',
        fp16=True,
        logging_steps=10,
        save_steps=50,
        save_total_limit=2,
        optim='adamw_8bit',
        weight_decay=0.01,
        report_to='none',
        seed=42,
    ),
)

print('Starting QLoRA fine-tuning (~30-40 min on T4)...')
trainer.train()

In [ ]:
# Save LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'LoRA adapter saved to {OUTPUT_DIR}')

In [ ]:
# Export to GGUF for Ollama (q4_k_m = best quality/size tradeoff)
gguf_path = OUTPUT_DIR + '/model.gguf'
print('Exporting to GGUF format (takes 5-10 min)...')
model.save_pretrained_gguf(OUTPUT_DIR, tokenizer, quantization_method='q4_k_m')
print(f'GGUF saved: {gguf_path}')

In [ ]:
# Download the GGUF file to your PC
from google.colab import files
import os

# Zip the output directory for download
!zip -r /content/naukri-cv-parser-lora.zip {OUTPUT_DIR}
files.download('/content/naukri-cv-parser-lora.zip')

print('\n=== DONE ===')
print('After downloading:')
print('1. Extract to ai-engine/models/naukri-cv-parser-lora/')
print('2. Copy the .gguf file to ai-engine/setup/')
print('3. Run: ollama create naukri-cv-parser -f ai-engine/setup/Modelfile')
print('4. Set OLLAMA_MODEL=naukri-cv-parser in your .env')

## After Training — Integrate with Ollama

1. Extract downloaded zip → `ai-engine/models/naukri-cv-parser-lora/`
2. Update `ai-engine/setup/Modelfile`:
   ```
   FROM ./models/naukri-cv-parser-lora/model.gguf
   PARAMETER temperature 0.1
   PARAMETER num_predict 4096
   ```
3. Create model in Ollama: `ollama create naukri-cv-parser -f ai-engine/setup/Modelfile`
4. Set env: `OLLAMA_MODEL=naukri-cv-parser` in your `.env`
5. Restart the AI engine